# 第 9 章：XVA，以及它背后的风险暴露计算

本笔记本不止于复现正文。第 1 部分重建本章的每一个数字；
从第 2 部分起，**用一个期限结构模型把敞口曲线算出来**，
而不是把正文那条平滑曲线当作给定，然后再拿它去做真实 XVA 台会做的事。

| 部分 | 做什么 |
|---|---|
| 1 | 例 9.1 至 9.6 与表 9.1，用正文自己的敞口曲线 |
| 2 | 一个 Hull–White 模型：模拟这笔互换，由路径得出 EE 与 PFE |
| 3 | 抵押品：起点金额、最小转让金额、保证金风险期 |
| 4 | 净额结算，以及一个净额结算组合值多少 |
| 5 | 错向风险，用一个违约强度模型 $\lambda(t)=e^{a+bX(t)}$ |
| 6 | 在模拟出来的敞口曲线上重建整套 XVA |

正文的敞口曲线是 $2.904\sqrt{t}\,(5-t)/5$，即扩散与摊还相互对抗，写成一条平滑曲线。
**模拟出来的敞口曲线并不平滑**，而第 2 部分展示了这份平滑掩盖了什么。

用 NumPy 与 SciPy。

表头保留英文：中文字符在等宽终端里是双宽的，而 Python 的 `%-46s` 按字符数计。

In [1]:
import numpy as np
from math import exp, sqrt
from scipy.optimize import brentq

NOTIONAL, T = 100.0, 5.0            # 百万美元、年
SPREAD, RECOVERY = 0.0120, 0.40     # 交易对手 CDS 利差、回收率
LAMBDA = SPREAD / (1 - RECOVERY)    # 信用三角：每年 2%
R_DISC, FUNDING, HURDLE = 0.04, 0.0060, 0.10


def show(label, got, printed, tol=6e-3):
    """把算出来的值印在本章所印的值旁边。"""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-46s %14.4f   book %12.4f   %s" % (label, got, printed, flag))


def EE_chapter(t):
    """正文那条示意曲线，单位百万美元：sqrt(t) 的扩散，乘 (T-t) 的摊还。"""
    return 2.904 * np.sqrt(t) * (T - t) / T


DURATION = (1 - exp(-R_DISC * T)) / R_DISC
show("discount-weighted duration", DURATION, 4.53)

discount-weighted duration                             4.5317   book       4.5300   ok


## 1. 正文的数字：例 9.1 至 9.6

预期风险暴露用来定价；潜在未来风险暴露用来设限额。它们回答的是不同的问题，
因此一位报 1030 万美元的信用审批人员与一位报 250 万美元的交易员并没有分歧。

In [2]:
peak_t = 5.0 / 3.0
sigma = 2.5 / 0.3989
show("peak expected exposure, $m", EE_chapter(peak_t), 2.5)
show("implied sigma, $m", sigma, 6.27)
show("95% 的 PFE，百万美元", 1.645 * sigma, 10.3)
show("PFE / EE", 1.645 * sigma / 2.5, 4.12, tol=2e-2)

peak expected exposure, $m                             2.4994   book       2.5000   ok
implied sigma, $m                                      6.2672   book       6.2700   ok
95% 的 PFE，百万美元                                        10.3096   book      10.3000   ok
PFE / EE                                               4.1238   book       4.1200   ok


In [3]:
mid = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
ee = EE_chapter(mid)
q = np.array([exp(-LAMBDA * (i - 1)) - exp(-LAMBDA * i) for i in range(1, 6)])
cover = (1 - RECOVERY) * q            # 为一美元风险暴露投保一年的成本
df = np.exp(-R_DISC * mid)

print("%6s %12s %16s %18s %16s" % ("year", "EE ($m)", "cost of cover", "discount factor",
                                   "contribution"))
for i in range(5):
    print("%6d %12.3f %15.3f%% %18.4f %16.5f" % (i + 1, ee[i], 100 * cover[i], df[i],
                                                 ee[i] * cover[i] * df[i]))
CVA = float((ee * cover * df).sum())
print("%6s %12s %16s %18s %16.5f" % ("", "", "", "CVA", CVA))

show("CVA, $", 1e6 * CVA, 94_000)
show("CVA, bp of notional", 1e4 * CVA / NOTIONAL, 9.4)
# 第 24 章的读法（违约损失率乘以违约概率）必须给出同一个数
show("CVA as (1-R) x sum EE q DF, $", 1e6 * (1 - RECOVERY) * float((ee * q * df).sum()), 94_000)

  year      EE ($m)    cost of cover    discount factor     contribution
     1        1.848           1.188%             0.9802          0.02152
     2        2.490           1.165%             0.9418          0.02731
     3        2.296           1.141%             0.9048          0.02371
     4        1.630           1.119%             0.8694          0.01585
     5        0.616           1.097%             0.8353          0.00564
                                                    CVA          0.09404
CVA, $                                             94037.0293   book   94000.0000   ok
CVA, bp of notional                                    9.4037   book       9.4000   ok
CVA as (1-R) x sum EE q DF, $                      94037.0293   book   94000.0000   ok


In [4]:
epe_discounted = float((ee * df).sum())
show("discounted expected exposure, $m-years", epe_discounted, 8.17)
FVA = FUNDING * epe_discounted
show("FVA, $", 1e6 * FVA, 49_000, tol=1e-2)

MVA = FUNDING * 1.25 * DURATION                   # 平均初始保证金 125 万美元
show("MVA, $", 1e6 * MVA, 34_000)

THRESH_EXPOSURE = 1.4                             # 剩下的平均未抵押风险暴露
show("CVA on a $5m threshold, $", 1e6 * (1 - RECOVERY) * LAMBDA * THRESH_EXPOSURE * DURATION,
     76_000)
show("funding on the threshold, $", 1e6 * FUNDING * THRESH_EXPOSURE * DURATION, 38_000)
show("total cost of the threshold, $",
     1e6 * ((1 - RECOVERY) * LAMBDA + FUNDING) * THRESH_EXPOSURE * DURATION, 114_000)

KVA = HURDLE * 0.4 * DURATION                     # 平均监管资本 40 万美元
show("KVA, $", 1e6 * KVA, 181_000)

discounted expected exposure, $m-years                 8.1650   book       8.1700   ok
FVA, $                                             48990.0148   book   49000.0000   ok
MVA, $                                             33987.9838   book   34000.0000   ok
CVA on a $5m threshold, $                          76133.0837   book   76000.0000   ok
funding on the threshold, $                        38066.5419   book   38000.0000   ok
total cost of the threshold, $                    114199.6256   book  114000.0000   ok
KVA, $                                            181269.2469   book  181000.0000   ok


## 表 9.1，以及 DVA 数字意味着什么

正文的表给出 DVA 为 $+39{,}000$，并说明交易商自身的利差取 49 个基点。
这一利差也可以单从 DVA 倒推出来，从而让两者互相核对：DVA 就是用交易商自身的违约强度、
按负风险暴露算出的 CVA，而对一笔平价互换，负风险暴露的曲线按对称性等于正风险暴露的曲线。

In [5]:
def dva_at(hazard):
    qo = np.array([exp(-hazard * (i - 1)) - exp(-hazard * i) for i in range(1, 6)])
    return (1 - RECOVERY) * float((ee * qo * df).sum())


own_hazard = brentq(lambda h: 1e6 * dva_at(h) - 39_000, 1e-6, 0.10)
DVA = dva_at(own_hazard)
print("the dealer's own hazard implied by DVA = +39,000 is %.3f%%," % (100 * own_hazard))
print("i.e. a CDS spread of about %.0f bp at %.0f%% recovery"
      % (1e4 * own_hazard * (1 - RECOVERY), 100 * RECOVERY))

print()
print("%-8s %18s %14s" % ("", "uncollateralized", "cleared"))
# 清算后的资本约 28,000 美元：例 27.3 的 EAD 0.929m x 3% 杠杆率
rows = [("CVA", -CVA * 1e6, 0.0), ("DVA", DVA * 1e6, 0.0), ("FVA", -FVA * 1e6, 0.0),
        ("MVA", None, -MVA * 1e6), ("KVA", -KVA * 1e6, -HURDLE * 28_000 * DURATION)]
fmt = lambda v: "---" if v is None else "{:+,.0f}".format(v)
for name, u, c in rows:
    print("%-8s %18s %14s" % (name, fmt(u), fmt(c)))
tot_u = -CVA * 1e6 + DVA * 1e6 - FVA * 1e6 - KVA * 1e6
tot_c = -MVA * 1e6 - HURDLE * 28_000 * DURATION
print("%-8s %18s %14s" % ("total", fmt(tot_u), fmt(tot_c)))
show("uncollateralized total, $", tot_u, -285_000)
show("cleared total, $", tot_c, -47_000, tol=1e-2)

the dealer's own hazard implied by DVA = +39,000 is 0.809%,
i.e. a CDS spread of about 49 bp at 40% recovery

           uncollateralized        cleared
CVA                 -94,037             +0
DVA                 +39,000             +0
FVA                 -48,990             +0
MVA                     ---        -33,988
KVA                -181,269        -12,689
total              -285,296        -46,677
uncollateralized total, $                        -285296.2911   book -285000.0000   ok
cleared total, $                                  -46676.8311   book  -47000.0000   ok


## 2. 把风险暴露算出来，而不是假定它

一个建在平坦初始曲线上的 Hull–White 短期利率、一笔五年期平价付固定互换、
四万条对偶路径。先做两项检查：这笔互换在起始时必须价值为零，
而模拟出来的债券价格必须复现初始曲线。

**然后看形状**。正文那条曲线是平滑的。模拟出来的曲线呈锯齿状：每付出一笔票息，风险暴露就立即下降，
因此这条曲线是付息日之间的一串小驼峰，而不是一个大驼峰。

In [6]:
A_MR, SIG_R, R0 = 0.05, 0.010, 0.04          # 均值回复、短期利率波动率、平坦曲线
PAY_TIMES = np.arange(0.5, T + 1e-9, 0.5)


def B(tau):
    return (1 - np.exp(-A_MR * tau)) / A_MR


def par_rate():
    annuity = sum(0.5 * exp(-R0 * t) for t in PAY_TIMES)
    return (1 - exp(-R0 * T)) / annuity


K_SWAP = par_rate()
print("平坦 %.1f%% 曲线上的平价互换利率: %.4f%%" % (100 * R0, 100 * K_SWAP))


def simulate(n_paths=20_000, n_steps=60, seed=3):
    """Hull--White 路径，以及付固定互换在每个日期上的价值。

    在两个重置日之间，当期的浮动支付「已经定盘」，因此重置日公式
    N[(1-P(t,T_n)) - K A(t)] 在那里是错的。这里逐路径携带最后一次定盘并显式估值；
    没有它，敞口曲线会在每个付息日朝错误的方向跳。
    """
    dt = T / n_steps
    delta = 0.5
    rng = np.random.default_rng(seed)
    z = rng.standard_normal((n_paths, n_steps))
    z = np.concatenate([z, -z])                   # 对偶变量
    m = len(z)
    x = np.zeros(m)
    L_fixed = np.full(m, K_SWAP)                  # 第一期在 t=0 定盘的利率
    grid, values, factor = [], [], []

    def bond(t, u, x):
        Vt = SIG_R ** 2 * (1 - exp(-2 * A_MR * t)) / (2 * A_MR) if t > 0 else 0.0
        return np.exp(-R0 * (u - t) - B(u - t) * x - 0.5 * B(u - t) ** 2 * Vt)

    for i in range(n_steps):
        t_prev = i * dt
        x = x * exp(-A_MR * dt) + SIG_R * sqrt((1 - exp(-2 * A_MR * dt)) / (2 * A_MR)) * z[:, i]
        t = (i + 1) * dt
        # 若跨过了一个重置日，新一期的利率就在那里定盘
        for tr in PAY_TIMES[:-1]:
            if t_prev < tr - 1e-9 <= t - 1e-9 or (t_prev < tr <= t):
                P_next = bond(tr, tr + delta, x)
                L_fixed = (1.0 / P_next - 1.0) / delta
        grid.append(t)
        factor.append(x.copy())
        future = PAY_TIMES[PAY_TIMES > t + 1e-9]
        if len(future) == 0:
            values.append(np.zeros(m))
            continue
        T_j, T_n = future[0], future[-1]
        P = np.array([bond(t, u, x) for u in future])
        floating = delta * L_fixed * P[0] + (P[0] - P[-1] if len(future) > 1 else 0.0)
        fixed = K_SWAP * delta * P.sum(axis=0)
        values.append(NOTIONAL * (floating - fixed))
    return np.array(grid), np.array(values), np.array(factor)


grid, V, X = simulate()
EE_mc = np.maximum(V, 0).mean(axis=1)
PFE_mc = np.quantile(np.maximum(V, 0), 0.95, axis=1)

print("第一步上的价值均值 %.4f （平价互换起始为零）" % V[0].mean())
print("peak EE %.3f at t = %.2f;  peak PFE %.3f" % (EE_mc.max(), grid[EE_mc.argmax()],
                                                    PFE_mc.max()))
print("峰值处的 PFE / EE: %.2f （例 9.1 由正态近似得到 %.2f）"
      % (PFE_mc[EE_mc.argmax()] / EE_mc[EE_mc.argmax()], 10.3 / 2.5))

平坦 4.0% 曲线上的平价互换利率: 4.0403%


第一步上的价值均值 0.0001 （平价互换起始为零）
peak EE 1.595 at t = 1.92;  peak PFE 6.494
峰值处的 PFE / EE: 4.07 （例 9.1 由正态近似得到 4.12）


In [7]:
print("模拟曲线与正文平滑曲线的对比，已按同一峰值缩放")
print("%8s %14s %16s %14s" % ("t", "simulated", "chapter", "difference"))
scale = 2.5 / EE_mc.max()
for t in (0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5):
    j = int(np.argmin(abs(grid - t)))
    a, b = scale * EE_mc[j], EE_chapter(t)
    print("%8.2f %14.3f %16.3f %14.3f" % (t, a, b, a - b))
print()
print("锯齿：每个付息日前后紧邻两点的 EE")
for t_pay in (1.0, 2.0, 3.0, 4.0):
    j1 = int(np.argmin(abs(grid - t_pay)))         # the payment has settled by this point
    j0 = j1 - 1                                    # the step before it
    print("  t=%.1f   before %.3f   after %.3f   drop %.1f%%"
          % (t_pay, scale * EE_mc[j0], scale * EE_mc[j1],
             100 * (1 - EE_mc[j1] / EE_mc[j0])))

模拟曲线与正文平滑曲线的对比，已按同一峰值缩放
       t      simulated          chapter     difference
    0.25          1.135            1.379         -0.245
    0.50          1.627            1.848         -0.221
    1.00          2.059            2.323         -0.264
    1.50          2.243            2.490         -0.247
    2.00          2.228            2.464         -0.236
    2.50          2.109            2.296         -0.187
    3.00          1.875            2.012         -0.137
    3.50          1.520            1.630         -0.109
    4.00          1.095            1.162         -0.066
    4.50          0.588            0.616         -0.028

锯齿：每个付息日前后紧邻两点的 EE
  t=1.0   before 2.130   after 2.059   drop 3.3%
  t=2.0   before 2.500   after 2.228   drop 10.9%
  t=3.0   before 2.282   after 1.875   drop 17.8%
  t=4.0   before 1.597   after 1.095   drop 31.4%


## 3. 抵押品：起点金额、最小转让金额与保证金风险期

CSA 并不消除风险暴露；它消除的是起点金额之上的部分，而且要等一段时间才生效。
**保证金风险期**是交易对手最后一次履行追缴与终止平仓之间的窗口，
非集中清算合约通常是十天。无论 CSA 怎么写，这段窗口内的风险暴露都是未抵押的。

例 9.5 把一个 500 万美元的起点金额定价为约 11.4 万美元（CVA 与 FVA 合计）。
这里由模拟路径算出，而不是假定。

In [8]:
def collateralized_exposure(V, grid, threshold, mta=0.25, mpor_days=10):
    """CSA 之后的风险暴露：抵押品带着滞后与起点金额去追随价值。"""
    mpor = mpor_days / 250.0
    lag = max(1, int(round(mpor / (grid[1] - grid[0]))))
    posted = np.zeros_like(V)
    for i in range(len(grid)):
        target = np.maximum(V[i] - threshold, 0.0)
        prev = posted[i - 1] if i else np.zeros(V.shape[1])
        move = np.where(np.abs(target - prev) >= mta, target - prev, 0.0)
        posted[i] = prev + move
    held = np.vstack([np.zeros((lag, V.shape[1])), posted[:-lag]])   # 抵押品是陈旧的
    return np.maximum(V - held, 0.0)


def cva_from_paths(exposure, grid, hazard=LAMBDA, recovery=RECOVERY):
    ee_t = exposure.mean(axis=1)
    dt = grid[1] - grid[0]
    surv = np.exp(-hazard * grid)
    dq = np.concatenate([[1 - surv[0]], surv[:-1] - surv[1:]])
    return (1 - recovery) * float((ee_t * dq * np.exp(-R_DISC * grid)).sum())


base = np.maximum(V, 0.0)
print("%14s %16s %16s %14s" % ("threshold ($m)", "average EE", "CVA ($)", "vs no CSA"))
uncol = cva_from_paths(base, grid)
for th in (np.inf, 10.0, 5.0, 2.0, 0.0):
    ex = base if np.isinf(th) else collateralized_exposure(V, grid, th)
    c = cva_from_paths(ex, grid)
    label = "none" if np.isinf(th) else "%.0f" % th
    print("%14s %16.3f %16s %13.0f%%" % (label, ex.mean(axis=1).mean(),
                                         "{:,.0f}".format(1e6 * c), 100 * c / uncol))
print()
print("起点金额为零并不给出零 CVA：保证金风险期仍在")

threshold ($m)       average EE          CVA ($)      vs no CSA
          none            1.094           57,458           100%
            10            1.093           57,401           100%
             5            1.025           53,801            94%
             2            0.684           35,842            62%
             0            0.126            6,917            12%

起点金额为零并不给出零 CVA：保证金风险期仍在


## 4. 净额结算

与同一交易对手的两笔交易。在净额结算协议之下，风险暴露是 $\max(\sum V_i, 0)$；
没有协议则是 $\sum \max(V_i, 0)$，而这两者之差正是 ISDA 终止净额结算存在的全部理由。

这里第二笔交易是与之相反的收固定互换，这是最极端的情形，净额结算几乎消去了全部风险暴露。

In [9]:
V2 = -0.7 * V                                  # 一笔相反方向的收固定互换，规模为 70%
gross = np.maximum(V, 0.0) + np.maximum(V2, 0.0)
netted = np.maximum(V + V2, 0.0)

cva_gross = cva_from_paths(gross, grid)
cva_net = cva_from_paths(netted, grid)
print("平均风险暴露，总额法  %.3f 百万美元" % gross.mean(axis=1).mean())
print("平均风险暴露，净额法  %.3f 百万美元" % netted.mean(axis=1).mean())
print("CVA gross  {:,.0f}".format(1e6 * cva_gross))
print("CVA netted {:,.0f}".format(1e6 * cva_net))
print("净额结算效益 %.0f%%" % (100 * (1 - cva_net / cva_gross)))

平均风险暴露，总额法  1.858 百万美元
平均风险暴露，净额法  0.328 百万美元
CVA gross  97,611
CVA netted 17,237
净额结算效益 82%


## 5. 错向风险

第 9.9 节提到违约强度模型：让交易对手的违约速率随风险暴露的驱动因子升降。
标准形式是 $\lambda(t) = e^{a + bX(t)}$，其中 $b>0$。
用到这里，$X$ 就是 Hull–White 因子：付固定互换在利率上升时获利，
因此 $b>0$ 使交易对手最可能违约的时点，恰好是交易商被欠得最多的时点。

每个日期的参数 $a$ 都经过设定，使**边际**违约概率保持不变。这一点很重要：
否则把 $b$ 调大也会抬高整体违约率，两种效应就混在一起了。
固定住边际概率就把相依性单独隔离出来，剩下的正是以违约为条件的风险暴露；
第 9.9 节说，标准计算用无条件平均代替了它。

In [10]:
def cva_wrong_way(V, X, grid, b, hazard=LAMBDA, recovery=RECOVERY):
    """lambda(t) = exp(a + b X(t))，其中 a 的取法使每个日期上 E[lambda] = hazard。"""
    dt = grid[1] - grid[0]
    tilt = np.exp(b * X)
    tilt = tilt / tilt.mean(axis=1, keepdims=True)      # E[tilt] = 1，因此边际概率不变
    lam = hazard * tilt
    surv = np.exp(-np.cumsum(lam, axis=0) * dt)
    prev = np.vstack([np.ones((1, V.shape[1])), surv[:-1]])
    dq = prev - surv                                    # 每一步内的违约概率
    ex = np.maximum(V, 0.0)
    return (1 - recovery) * float((ex * dq * np.exp(-R_DISC * grid)[:, None]).mean(axis=1).sum())


independent = cva_wrong_way(V, X, grid, 0.0)
print("%10s %18s %16s" % ("b", "CVA ($)", "vs independent"))
for b in (-40.0, -20.0, 0.0, 20.0, 40.0, 80.0):
    c = cva_wrong_way(V, X, grid, b)
    tag = "right-way" if b < 0 else ("independent" if b == 0 else "wrong-way")
    print("%10.0f %18s %14.2fx   %s" % (b, "{:,.0f}".format(1e6 * c),
                                        c / independent, tag))
print()
print("在错向情形下，独立性假设给出的是一个下界，正如第 9.9 节所说")
print("监管的做法是一个统一的 1.4 乘数（第 27.5 节）")

         b            CVA ($)   vs independent


       -40             28,240           0.49x   right-way


       -20             40,954           0.71x   right-way


         0             57,458           1.00x   independent


        20             77,567           1.35x   wrong-way


        40            100,312           1.75x   wrong-way


        80            144,092           2.51x   wrong-way

在错向情形下，独立性假设给出的是一个下界，正如第 9.9 节所说
监管的做法是一个统一的 1.4 乘数（第 27.5 节）


## 6. 整套 XVA，建在模拟出来的敞口曲线上

用路径而不是正文的曲线重建表 9.1。数字不会分毫不差（模拟出来的曲线峰值不同，而且带锯齿），
但**次序必须保住**，而正文的论证依据的正是这个次序：KVA 最大，其次 CVA，再次 FVA，最后 MVA；
而在集中清算之下，MVA 以小得多的成本取代了 CVA 与 FVA。

In [11]:
ee_t = np.maximum(V, 0.0).mean(axis=1)
dt = grid[1] - grid[0]
disc = np.exp(-R_DISC * grid)
scale = 2.5 / ee_t.max()                        # 换到正文的尺度上以便比较

cva_sim = scale * cva_from_paths(np.maximum(V, 0.0), grid)
fva_sim = FUNDING * scale * float((ee_t * disc * dt).sum())
dva_sim = scale * cva_from_paths(np.maximum(V, 0.0), grid, hazard=own_hazard)
kva_sim = HURDLE * 0.4 * DURATION
mva_sim = FUNDING * 1.25 * DURATION

print("%-8s %20s %20s" % ("", "simulated ($)", "chapter ($)"))
for name, sim, book in (("CVA", -cva_sim, -CVA), ("DVA", dva_sim, DVA),
                        ("FVA", -fva_sim, -FVA), ("KVA", -kva_sim, -KVA)):
    print("%-8s %20s %20s" % (name, "{:+,.0f}".format(1e6 * sim), "{:+,.0f}".format(1e6 * book)))
print("%-8s %20s %20s" % ("total",
                          "{:+,.0f}".format(1e6 * (-cva_sim + dva_sim - fva_sim - kva_sim)),
                          "{:+,.0f}".format(1e6 * (-CVA + DVA - FVA - KVA))))
print()
order = sorted([("CVA", cva_sim), ("FVA", fva_sim), ("KVA", kva_sim), ("MVA", mva_sim)],
               key=lambda p: -p[1])
print("由大到小: " + " > ".join(n for n, _ in order))

                simulated ($)          chapter ($)
CVA                   -90,063              -94,037
DVA                   +37,400              +39,000
FVA                   -47,021              -48,990
KVA                  -181,269             -181,269
total                -280,954             -285,296

由大到小: KVA > CVA > FVA > MVA
